# Unit Responses to Playback Stimuli

In [3]:
session_dir = "../data/birds/GreYel3594M/GreYel3594M___170919_094646"

In [4]:
bird, session = session_dir.split("/")[-2:]

In [19]:
# import cPickle as pickle
try:
    import cPickle as pickle
except ImportError:
    import pickle

import matplotlib.pyplot as plt
import numpy as np
import os

from segment_analysis import MultiSegment
from playbacks_analysis import PlaybacksManager
from figure_scripts.unit_summary import get_square_dims, plot_waveforms

%load_ext autoreload
%autoreload 2
import config

In [83]:
from ipywidgets import interact, widgets
from IPython.display import clear_output
%matplotlib notebook

In [7]:
data_path = os.path.join(session_dir, "spikes", "{}-sorted-resegmented.pkl").format(session)

In [8]:
with open(data_path, "rb") as datafile:
    block = pickle.load(datafile)
block.create_relationship(force=True, append=True, recursive=True)

In [20]:
playbacks_manager = PlaybacksManager(block.segments)
playback_multisegs = playbacks_manager.group_playback_responses()
multisegs = dict((stim_id, playback_multisegs[stim_id]) for stim_id in playbacks_manager.stim_ids)

In [94]:
stim_dropdown = widgets.Dropdown(
    options=[
        ("{vocalizer}, {vocalization_type}: {stim}".format(stim=stim, **multisegs[stim].annotations), playback_multisegs[stim])
            for stim in playbacks_manager.stim_ids],
    value=playback_multisegs[playbacks_manager.stim_ids[0]],
    description="Stim",
    disabled=False
)

electrode_dropdown = widgets.Dropdown(
    options=[(c.name, c) for c in block.channel_indexes[:-1]],
    value=block.channel_indexes[0],
    description='Electrode:',
    disabled=False,
)

unit_dropdown = widgets.Dropdown(
    options=[("--", None)] + [(e.name, e) for e in electrode_dropdown.value.units],
    value=electrode_dropdown.value.units[0],
    description='Unit:',
    disabled=False,
)

o = widgets.Output()
o2 = widgets.Output()

def get_dims(n):
    cols = 2
    rows = int(np.ceil(n / float(cols)))
    return cols, rows

def select_electrode(elec):
    elec = elec["owner"]
    elec = elec.options[elec.index][1]
    with o:
        clear_output()
    unit_dropdown.options = [("--", None)] + [(e.name, e) for e in elec.units]

all_units = [unit for elec in block.channel_indexes[:-1] for unit in elec.units]
def get_unit_idx(unit):
    for unit_idx, u in enumerate(all_units):
        if unit == u:
            break
    return unit_idx

def select_unit(eh):
    u = unit_dropdown.value
    if u is None:
        return
    
    with o:
        clear_output()
#         w, h = get_dims(len(stims))

        multiseg = stim_dropdown.value
        
        fig, ax = plt.subplots(1, 1, figsize=(multiseg.t_stop * 3, 4))
        
        multiseg.plotter.plot_spectrogram(multiseg.segments[0], signal="playback", channel=None, ax=ax)

        ax.set_ylabel("")
        ax.set_xlabel("")
        ax.set_ylim(-ax.get_ylim()[1], ax.get_ylim()[1])
        ax.set_xticks([])
        ax.set_yticks([])
        
        raster_ax = ax.twinx()
        unit_idx = get_unit_idx(u)
        
        multiseg.plotter.plot_raster(unit_idx, ax=raster_ax, mark_noise_epochs=True, elec_idx=electrode_dropdown.value.index)
        raster_ax.set_xlim(0, multiseg.t_stop)
        raster_ax.set_xticks([])
        raster_ax.set_yticks([])
        raster_ax.set_ylim(len(multiseg.segments), -len(multiseg.segments))
        
#         psth_ax = ax.twinx()
#         multiseg.plotter.plot_psth(unit_idx, ax=psth_ax, alpha=0.0, mode="gaussian", convolve_params=(10,))

        plt.show()
        
    with o2:
        clear_output()
        fig, ax = plt.subplots(1, 1, figsize=(4, 2))
        ax.plot(multiseg.waveforms(unit=unit_idx)[0].T, color="Blue", alpha=0.1)
        plt.show()
        
    for seg in multiseg:
        print(seg.t_start, seg.t_stop)

stim_dropdown.observe(select_unit, "value", "change")
    
electrode_dropdown.observe(select_electrode, "value", "change")
unit_dropdown.observe(select_unit, "value", "change")
    
display(widgets.VBox([
    widgets.HBox([electrode_dropdown, unit_dropdown, stim_dropdown]),
    widgets.VBox([o]),
    widgets.VBox([o2])
]))

HBox(children=(VBox(children=(Dropdown(description=u'Electrode:', options=(('Amp Elec 5', <neo.core.channelindex.ChannelIndex object at 0x7f093b499490>), ('Amp Elec 10', <neo.core.channelindex.ChannelIndex object at 0x7f093b499510>), ('Amp Elec 12', <neo.core.channelindex.ChannelIndex object at 0x7f093b4995d0>), ('Amp Elec 13', <neo.core.channelindex.ChannelIndex object at 0x7f093b499690>)), value=<neo.core.channelindex.ChannelIndex object at 0x7f093b499490>), Dropdown(description=u'Unit:', index=1, options=(('--', None), ('e5u0', <neo.core.unit.Unit object at 0x7f093b4994d0>)), value=<neo.core.unit.Unit object at 0x7f093b4994d0>), Output())), VBox(children=(Checkbox(value=True, description=u'con_female, DC: GreHpi3607M_FAF_DC_0.wav'), Checkbox(value=True, description=u'con_female, DC: GreHpi3607M_FAF_DC_1.wav'), Checkbox(value=True, description=u'con_female, DC: GreHpi3607M_FAF_DC_2.wav'), Checkbox(value=True, description=u'con_female, Te: GreHpi3607M_FAF_Te_0.wav'), Checkbox(value=True, description=u'con_female, Te: GreHpi3607M_FAF_Te_1.wav'), Checkbox(value=True, description=u'con_female, Te: GreHpi3607M_FAF_Te_2.wav'), Checkbox(value=True, description=u'con_male, DC: GreHpi3607M_MAF_DC_0.wav'), Checkbox(value=True, description=u'con_male, DC: GreHpi3607M_MAF_DC_1.wav'), Checkbox(value=True, description=u'con_male, So: GreHpi3607M_MAF_So_0.wav'), Checkbox(value=True, description=u'con_male, So: GreHpi3607M_MAF_So_1.wav'), Checkbox(value=True, description=u'con_male, So: GreHpi3607M_MAF_So_2.wav'), Checkbox(value=True, description=u'con_male, So: GreHpi3607M_MAF_So_3.wav'), Checkbox(value=True, description=u'con_male, Te: GreHpi3607M_MAF_Te_0.wav'), Checkbox(value=True, description=u'tutor, DC: GreHpi3607M_MAT_DC_0.wav'), Checkbox(value=True, description=u'tutor, So: GreHpi3607M_MAT_So_0.wav'), Checkbox(value=True, description=u'tutor, So: GreHpi3607M_MAT_So_1.wav'), Checkbox(value=True, description=u'tutor, So: GreHpi3607M_MAT_So_2.wav'), Checkbox(value=True, description=u'tutor, So: GreHpi3607M_MAT_So_3.wav'), Checkbox(value=True, description=u'tutor, Te: GreHpi3607M_MAT_Te_0.wav'), Checkbox(value=True, description=u'tutor, Te: GreHpi3607M_MAT_Te_1.wav'), Checkbox(value=True, description=u'tutor, Te: GreHpi3607M_MAT_Te_2.wav'), Checkbox(value=True, description=u'con_male, So: GreHpi3607M_MJF_So_0.wav'), Checkbox(value=True, description=u'con_male, So: GreHpi3607M_MJF_So_1.wav'), Checkbox(value=True, description=u'con_male, So: GreHpi3607M_MJF_So_2.wav')))))